In [1]:
from code import *
import tensorflow as tf
import numpy as np
import os

In [2]:
os.chdir('../../')  # change dir to the work root DEDL-Kcr

In [3]:
 # 1.load_data
test_filepath = r'dataset/test_dataset.csv'
test_seqs, y_test = load_dataset(test_filepath)

# 2.extrac features
protein_bert_test = extract_embedding_features(test_seqs)
tf.keras.backend.clear_session()
BLOSUM62_test = BLOSUM62(test_seqs)
one_hot_test = np.array(to_embedding_numeric(test_seqs)).astype(np.float32)

In [4]:
model_1 = CNN(protein_bert_test)
model_2 = BiGRU()
model_3 = CNN(BLOSUM62_test)
model_atnn = ensemble_model()

In [5]:
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')

In [6]:
ensemble_valid_input = np.concatenate((model_1.predict(protein_bert_test, verbose=0),
                                  model_2.predict(one_hot_test, verbose=0),
                                  model_3.predict(BLOSUM62_test, verbose=0)), axis=-1)

In [7]:
outputs = 'code/experiment/results/ind_test'
prediction_result_ind = []
p_1 = model_atnn.predict(ensemble_valid_input, verbose=0)
tmp_result = np.zeros((len(y_test), 3))
tmp_result[:, 0], tmp_result[:, 1:] = y_test, p_1
prediction_result_ind.append(tmp_result)
metrics = save_val_result(prediction_result_ind, outputs, "Independent_test")

# Papaya dataset

In [3]:
def parse_sequences(path):
    fr = open(path, 'r')
    data = fr.readlines()
    seqs = data[1::2]
    return seqs

def clean(x):
    return x.strip()

In [4]:
papaya_pos = parse_sequences("dataset/papaya_31_neg.txt")
papaya_neg = parse_sequences("dataset/papaya_31_pos.txt")

In [5]:
papaya_pos = list(map(clean, papaya_pos))
papaya_neg = list(map(clean, papaya_neg))

In [6]:
papaya_ds = papaya_pos + papaya_neg
labels = len(papaya_pos) * [0] + len(papaya_neg) * [1]

In [7]:
papaya_ds_fill = []
for seq in papaya_ds:
    if "-" in seq:
        seq = list(seq)
        for i in range(len(seq)):
            if seq[i] == "-":
                seq[i] = seq[30-i]
    papaya_ds_fill.append("".join(seq))

In [8]:
protein_bert_test = extract_embedding_features(papaya_ds_fill)
tf.keras.backend.clear_session()
BLOSUM62_test = BLOSUM62(papaya_ds_fill)
one_hot_test = np.array(to_embedding_numeric(papaya_ds_fill)).astype(np.float32)

In [9]:
model_1 = CNN(protein_bert_test)
model_2 = BiGRU()
model_3 = CNN(BLOSUM62_test)
model_atnn = ensemble_model()

In [10]:
model_1.load_weights('model/dedl_kcr/model1.h5')
model_2.load_weights('model/dedl_kcr/model2.h5')
model_3.load_weights('model/dedl_kcr/model3.h5')
model_atnn.load_weights('model/dedl_kcr/model_ensemble.h5')

In [14]:
labels = np.array(labels)
from sklearn.model_selection import StratifiedKFold
outputs = r'code/experiment/results/ind_test'
mkdir(outputs)
prediction_result_cv = []
folds = StratifiedKFold(5, shuffle=True).split(protein_bert_test, labels)
for i, (train, valid) in enumerate(folds):
    train_x_emb, train_y = protein_bert_test[train], labels[train]
    valid_x_emb, valid_y = protein_bert_test[valid], labels[valid]
    train_x_onehot, valid_x_onehot = one_hot_test[train], one_hot_test[valid]
    train_x_BLOSUM62, valid_x_BLOSUM62 = BLOSUM62_test[train], BLOSUM62_test[valid]
    p_1 = model_1.predict(valid_x_emb, verbose=0)
    p_2 = model_2.predict(valid_x_onehot, verbose=0)
    p_3 = model_3.predict(valid_x_BLOSUM62, verbose=0)
    p = tf.concat([p_1, p_2, p_3], axis=1)
    valid_p = model_atnn.predict(p, verbose=0)
    tmp_result = np.zeros((len(valid_y), 3))
    tmp_result[:, 0], tmp_result[:, 1:] = valid_y, valid_p
    prediction_result_cv.append(tmp_result)
    tf.keras.backend.clear_session()
out = os.path.join(outputs, "papaya")
mean_auc, aucs = plot_roc_curve(prediction_result_cv, out + '_roc_cv.svg', label_column=0, score_column=2)

(0.8469568437579512,
 [0.843295790599635,
  0.8412277942070933,
  0.8561337276369051,
  0.8541873780909834,
  0.8428029111348813])